# **作业2_1：音素分类**

幻灯片: https://speech.ee.ntu.edu.tw/~hylee/ml/ml2021-course-data/hw/HW02/HW02.pdf

## 解压数据

In [33]:
from google.colab import drive
drive.mount('/content/drive')

!_ZIP_PATH="/content/drive/MyDrive/ColabNotebooks/ML2021_HW2/ml2021spring-hw2.zip"; \
_TARGET="/content/timit_11"; \
if [ -f "$_TARGET/train_11.npy" ] && [ -f "$_TARGET/train_label_11.npy" ] && [ -f "$_TARGET/test_11.npy" ]; then \
    echo "✅ 数据已存在，跳过解压"; \
else \
    if [ -f "$_ZIP_PATH" ]; then \
        echo "⏳ 开始解压..."; \
        unzip -o "$_ZIP_PATH" -d /content; \
        if [ -d "$_TARGET/timit_11" ]; then \
            mv "$_TARGET"/timit_11/* "$_TARGET"/ 2>/dev/null; \
            rmdir "_$TARGET/timit_11" 2>/dev/null; \
        fi; \
        echo "✅ 解压完成"; \
    else \
        echo "❌ 未找到压缩包: $_ZIP_PATH"; \
    fi; \
fi; \
ls -lh "$_TARGET" 2>/dev/null || echo "⚠️ timit_11 目录不存在"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ 数据已存在，跳过解压
total 5.4G
-rw-r--r-- 1 root root 1.5G Mar  6  2021 test_11.npy
drwxr-xr-x 2 root root 4.0K Oct  2 07:04 timit_11
-rw-r--r-- 1 root root 4.0G Mar  6  2021 train_11.npy
-rw-r--r-- 1 root root 9.4M Mar  6  2021 train_label_11.npy


## 准备数据
从 `.npy` 文件（NumPy 数组）中加载训练和测试数据。

In [34]:
import numpy as np

print('Loading data ...')

data_root='./timit_11/'
train = np.load(data_root + 'train_11.npy')
train_label = np.load(data_root + 'train_label_11.npy')
test = np.load(data_root + 'test_11.npy')

print('Size of training data: {}'.format(train.shape))
print('Size of testing data: {}'.format(test.shape))

Loading data ...
Size of training data: (1229932, 429)
Size of testing data: (451552, 429)


## 创建数据集

In [35]:
import torch
from torch.utils.data import Dataset

class TIMITDataset(Dataset):
  # 在初始化该类时，把传入的 NumPy 数组（特征 X 和标签 y）转成 PyTorch 模型能处理的张量（Tensor）。
  def __init__(self, X, y=None):
    # 将 NumPy 数组转为 PyTorch 的 FloatTensor (32位浮点数)
    self.data = torch.from_numpy(X).float()
    if y is not None:
      # 将标签转换为整数类型
      y = y.astype(np.int64)
      # 转为 PyTorch 的 LongTensor (64位整数，用于分类任务的类别索引)
      self.label = torch.LongTensor(y)
    else:
      self.label = None

  def __getitem__(self, idx):
    if self.label is not None:
      # 训练集/验证集：返回第 idx 条特征和对应的标签
      return self.data[idx], self.label[idx]
    else:
      # 测试集：只返回第 idx 条特征
      return self.data[idx]

  def __len__(self):
    # 返回数据集的样本总数
    return len(self.data)


将标注数据分为训练集和验证集，你可以修改变量 `VAL_RATIO` 来调整验证数据的比例。

In [36]:
VAL_RATIO = 0.2

percent = int(train.shape[0] * (1 - VAL_RATIO))
train_x, train_y, val_x, val_y = train[:percent], train_label[:percent], train[percent:], train_label[percent:]
print('Size of training set: {}'.format(train_x.shape))
print('Size of validation set: {}'.format(val_x.shape))

Size of training set: (983945, 429)
Size of validation set: (245987, 429)


从数据集创建数据加载器，可在此处自由调整变量 `BATCH_SIZE`。

In [37]:
BATCH_SIZE = 64

from torch.utils.data import DataLoader

train_set = TIMITDataset(train_x, train_y)
val_set = TIMITDataset(val_x, val_y)
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False)

清理不必要的变量以节省内存。<br>

**注意：如果之后需要使用这些变量，可以移除此代码块或稍后清理不需要的变量**<br>
**数据量非常大，因此请注意 Colab 中的内存使用情况**


In [38]:
import gc

del train, train_label, train_x, train_y, val_x, val_y
gc.collect()

576

## 创建模型

定义模型架构，建议您根据需要进行修改和实验。

In [39]:
import torch
import torch.nn as nn

class Classifier(nn.Module):
  def __init__(self, dropout_rate=0.25):
    super(Classifier, self).__init__()

    # 使用 nn.Sequential 构建更深、表达能力更强的结构
    self.net = nn.Sequential(
        # Block 1
        nn.Linear(429, 2048),
        nn.BatchNorm1d(2048),
        nn.LeakyReLU(0.1),
        nn.Dropout(dropout_rate),

        # Block 2
        nn.Linear(2048, 1024),
        nn.BatchNorm1d(1024),
        nn.LeakyReLU(0.1),
        nn.Dropout(dropout_rate),

        # Block 3
        nn.Linear(1024, 512),
        nn.BatchNorm1d(512),
        nn.LeakyReLU(0.1),
        nn.Dropout(dropout_rate),

        # Block 4
        nn.Linear(512, 256),
        nn.BatchNorm1d(256),
        nn.LeakyReLU(0.1),
        nn.Dropout(dropout_rate),

        # Output Layer (输出 39 个类别的 raw logits)
        nn.Linear(256, 39)
    )

  def forward(self, x):
    return self.net(x)

## 训练

In [40]:
"""
自动选择计算设备
有 GPU 就返回 'cuda' 开启硬件加速；没有则退回 'cpu'。
"""
def get_device():
  return 'cuda' if torch.cuda.is_available() else 'cpu'

修复随机种子以确保可重现性。

In [41]:
# 固定所有随机种子
def same_seeds(seed):
  torch.manual_seed(seed)
  if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
  np.random.seed(seed)
  torch.backends.cudnn.benchmark = False
  torch.backends.cudnn.deterministic = True

请随意在此更改训练参数。

In [42]:
# 固定随机种子
same_seeds(0)
device = get_device()
print(f'DEVICE: {device}')

# 优化后的训练参数
num_epoch = 30
# 配合 BatchNorm，初始 LR 提至 0.001
learning_rate = 1e-3
# 增至 512 以加速 GPU 并行并稳定 Batch Normalization
batch_size = 512
model_path = './model.ckpt'

# 重新初始化 DataLoader
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False)

# 初始化模型与优化器
model = Classifier(dropout_rate=0.4).to(device)
criterion = nn.CrossEntropyLoss()

# AdamW 配合 weight_decay 控制权重大小，防过拟合
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-3)

# 余弦退火学习率调度器
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epoch, eta_min=1e-5)

DEVICE: cuda


In [43]:
# 记录历史最佳验证准确率，用于判断是否保存模型
best_acc = 0.0

for epoch in range(num_epoch):
  # 累加训练正确样本数与训练损失
  train_acc = 0.0
  train_loss = 0.0
  # 累加验证正确样本数与验证损失
  val_acc = 0.0
  val_loss = 0.0

  # 1. 训练阶段
  # 切换到训练模式（激活 Dropout 和 BatchNorm 的训练行为）
  model.train()
  for i, data in enumerate(train_loader):
    # inputs 是特征向量，labels 是真实音素类别
    inputs, labels = data
    # 将数据搬运到 GPU 或 CPU 设备
    inputs, labels = inputs.to(device), labels.to(device)

    # 清空上一轮累积的梯度
    optimizer.zero_grad()
    # 前向传播得到模型输出 (Logits)
    outputs = model(inputs)
    # 计算交叉熵损失
    batch_loss = criterion(outputs, labels)

    # 取预测概率最大的类别索引 (0~38)
    _, train_pred = torch.max(outputs, 1)
    # 反向传播，利用链式法则计算参数梯度
    batch_loss.backward()
    # 更新模型权重参数
    optimizer.step()

    # 直接在 GPU 上进行布尔比较并统计正确预测数，通过 .item() 提纯标量数值
    train_acc += (train_pred == labels).sum().item()
    # 累加当前 Batch 的损失值（断开计算图，避免显存泄漏）
    train_loss += batch_loss.item()

  # 训练完一个 Epoch 后，按策略更新学习率
  scheduler.step()

  # 2. 验证阶段
  if len(val_set) > 0:
    # 切换到评估模式（固定 BatchNorm 参数，关闭 Dropout 随机丢弃）
    model.eval()
    # 关闭梯度计算，节省显存并加快评估速度
    with torch.no_grad():
      for i, data in enumerate(val_loader):
        inputs, labels = data
        inputs, labels = inputs.to(device), labels.to(device)

        # 前向传播与损失计算
        outputs = model(inputs)
        batch_loss = criterion(outputs, labels)
        _, val_pred = torch.max(outputs, 1)

        # 累加验证集的正确预测数与损失
        val_acc += (val_pred == labels).sum().item()
        val_loss += batch_loss.item()

        # 计算整个 Epoch 的平均准确率与平均 Loss
        train_accuracy = train_acc / len(train_set)
        val_accuracy = val_acc / len(val_set)

      # 打印本轮结果
      print('[{:03d}/{:03d}] Train Acc: {:3.6f} Loss: {:3.6f} | Val Acc: {:3.6f} loss: {:3.6f}'.format(
        epoch + 1, num_epoch,
        # 训练准确率 = 正确数 / 总样本数
        train_acc/len(train_set),
        # 训练平均损失 = 总损失 / batch数
        train_loss/len(train_loader),
        # 验证准确率
        val_acc/len(val_set),
        # 验证平均损失
        val_loss/len(val_loader)
      ))

      # 如果验证集准确率突破新高，保存当前最佳模型权重（Checkpoint）
      if val_accuracy > best_acc:
          best_acc = val_accuracy
          torch.save(model.state_dict(), model_path)
          print(f'Saving model with acc {best_acc:.4f}')
  else:
    # 没有验证集，就不做验证，只打印训练信息。
    print('[{:03d}/{:03d}] Train Acc: {:3.6f} Loss: {:3.6f}'.format(
      epoch + 1, num_epoch, train_acc/len(train_set), train_loss/len(train_loader)
    ))

# 若未划分验证集，默认保存最后一轮训练的模型
if len(val_set) == 0:
  torch.save(model.state_dict(), model_path)
  print('Saving model at last epoch')

[001/030] Train Acc: 0.577135 Loss: 1.380126 | Val Acc: 0.673442 loss: 1.027817
Saving model with acc 0.6734
[002/030] Train Acc: 0.636110 Loss: 1.158735 | Val Acc: 0.698029 loss: 0.937139
Saving model with acc 0.6980
[003/030] Train Acc: 0.657883 Loss: 1.079813 | Val Acc: 0.708424 loss: 0.896751
Saving model with acc 0.7084
[004/030] Train Acc: 0.671482 Loss: 1.030821 | Val Acc: 0.716948 loss: 0.864510
Saving model with acc 0.7169
[005/030] Train Acc: 0.681754 Loss: 0.994779 | Val Acc: 0.723095 loss: 0.841983
Saving model with acc 0.7231
[006/030] Train Acc: 0.690153 Loss: 0.965104 | Val Acc: 0.730839 loss: 0.821221
Saving model with acc 0.7308
[007/030] Train Acc: 0.696521 Loss: 0.941937 | Val Acc: 0.731921 loss: 0.810700
Saving model with acc 0.7319
[008/030] Train Acc: 0.701311 Loss: 0.923810 | Val Acc: 0.735092 loss: 0.800034
Saving model with acc 0.7351
[009/030] Train Acc: 0.706354 Loss: 0.905734 | Val Acc: 0.737470 loss: 0.791952
Saving model with acc 0.7375
[010/030] Train Acc

## 测试

创建测试数据集，并从保存的检查点加载模型。

In [44]:
test_set = TIMITDataset(test, None)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False)

model = Classifier().to(device)
model.load_state_dict(torch.load(model_path))

<All keys matched successfully>

做出预测

In [45]:
predict = []
model.eval()
with torch.no_grad():
  for i, data in enumerate(test_loader):
    inputs = data
    inputs = inputs.to(device)
    outputs = model(inputs)
    _, test_pred = torch.max(outputs, 1)

    for y in test_pred.cpu().numpy():
      predict.append(y)

将预测结果写入CSV文件。

In [47]:
with open('prediction.csv', 'w') as f:
  f.write('Id,Class\n')
  for i, y in enumerate(predict):
    f.write('{},{}\n'.format(i, y))

from google.colab import files
files.download('prediction.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>